# v102_dense_tune — v101 with the rule tuned against a test-sized pool

| Field | Value |
|---|---|
| **Version** | `v102_dense_tune` |
| **Plan group** | E2 (integrated run, INT) |
| **Parent version** | v101 |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-25 |
| **Status** | shortlisted |

v101's trained matcher, unchanged; only the decision rule is re-tuned, on the same tune-side
entities but blocked against the whole train-fold pool (~the test pool's size). Every stage is library code in
`src/entity_resolution/` (tested, documented); this notebook only configures, runs and
inspects it:

```
raw TSV -> normalize (static rules + learned transliteration map) -> blocking (exact keys,
name+address word TF-IDF, short-address name char-grams, per country) -> features (47 pair
similarities, chunked) -> LightGBM matcher -> decision rule tuned for macro F0.5 on the tune
split (pool-side 1-to-1) -> matching_results.tsv + candidate_pairs.tsv
```

House rules: every code cell is preceded by a markdown cell saying what it does and why;
the only decision metric is macro F0.5 on the fixed validation fold.

## 1. Hypothesis

* **Change vs parent (v101):** the rule is re-tuned with `tune_pool="train"`: the 441k
  tune-side entities are blocked against every train-fold pool record (~8.3M, about the
  test pool's 10M) instead of their own fold's pool (~2.1M). The matcher, features, token map
  and every other stage are v101's.
* **Why:** the val and tune folds are 20 % samples, so each of their entities meets ~5x fewer
  same-name decoys than a test entity. A rule tuned there is tuned for a sparser world than
  the leaderboard's; tuning it where the decoy density matches test should trade a little
  val F0.5 for precision where it counts.
* **How it is judged:** plain val F0.5 cannot show the gain (val is the sparse world), so the
  decision uses the dense-val diagnostic too (val entities blocked against val + train
  pools): keep if dense-val F0.5 beats v101's there and plain val does not fall by more
  than 0.003.

## 2. Setup

Imports from the shared library, this experiment's folders and the pipeline configuration.
`PipelineConfig()` holds every choice of this version (its defaults *are* the V1 plan);
it is saved to `artifacts/config.json` and logged in `metrics.json`. `timings` collects the
stage run times under the eight standard labels (13 §2.2).

In [ ]:
import json
import subprocess
import sys
import time
from dataclasses import asdict

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import PASS_BITS
from entity_resolution.evaluate import error_samples, harder_fold, pair_in, slice_report
from entity_resolution.features import FEATURE_COLUMNS, feature_names
from entity_resolution.normalize import NORM_COLUMNS, normalise_records
from entity_resolution.pipeline import (
    PipelineConfig, fit, load_normalised, peak_rss_gb, run_fold, run_test,
)
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v102_dense_tune"
ARTIFACTS = EXP_DIR / "artifacts"
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.model import MatcherParams
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000), tune_pool="train")
timings: dict[str, float] = {}
t_start = time.time()
print(json.dumps(cfg.record(), indent=1)[:3000])

## 3. Data

The fixed validation split (`split.load_fold`, 20 % of Source 1 by id hash, seed 42): every
version scores the same held-out entities against the same pool, so local F0.5 values are
comparable. Everything trainable is fitted on the `train` fold only; inside it,
`trainset.inner_split` (seed 4242, 25 % tune) separates model training (fit side) from
early stopping and threshold tuning (tune side). The val pool keeps the records matched to
*other* val entities: they are the decoys that make singletons and same-name businesses hard.

In [ ]:
with timed("load", timings):
    train = load_fold("train", columns=[])   # ids only: the pipeline reads normalised records
    val = load_fold("val")                   # raw columns kept for the error samples
pd.DataFrame([train.summary(), val.summary()])

## 4. Method

### 4.1 Normalisation (`normalize.py`, `token_maps.py`)

Rules, in order (05 §2), each undoing a noise pattern measured on true pairs:

* **Transliteration** (`anyascii`, ISC): rows with non-ASCII characters are transliterated on
  the raw text (Indic scripts, accents). 7 % of pool names and 9 % of pool addresses are in an
  Indic script; Source 1 is all Latin.
* **Case, `&` → `and`, punctuation, apostrophes, dotted initials** (`L.L.C.` → `llc`).
* **Domain / handle forms** (`allh0spitalityproducts.com`, `@sakashpoint`,
  `ORTHOPEDICHEALTHCOM`) lose their suffix; **leet digits** inside words fold to letters
  (`F0rman` → `forman`).
* **Legal forms** leave `name_core` and are kept, canonical, in `legal_form` (`Pvt. Ltd.` =
  `Private Limited` = `pvt ltd`; SARL/SAS/EURL for France). **Honorific prefixes** injected
  into pool names (`Mr`, `Smt`, `Shri`) are dropped.
* **Learned transliteration map**: for true pairs of the train fold whose pool name is in an
  Indic script, tokens are aligned by position with the Latin name (`सॉल्यूशंस` →
  `solyusms` ↔ `solutions`); tokens seen ≥ 3 times with a ≥ 50 % consistent partner form a
  map applied to non-Latin names. Learned from the provided training pairs only.
* **Addresses**: region components (full name, code or native-script name: `Maharashtra`,
  `MH`, `महाराष्ट्र`) become one code; ordinals lose their suffix; digits split from letters;
  leading zeros go; street types are canonicalised (`Street`/`St`/`Saint` — the generator
  writes `Saint` for `St`); old/new Indian city names are unified.

Derived keys: `name_first`, `name_sorted` (sorted distinct tokens), `name_squash` (letters and
digits only), `addr_nums`, `postcode`, `region`, `name_addr`. The cell shows raw and
normalised val records, including Indic-script names (static rules only: the learned map is
fitted inside `pipeline.fit` in §4.4, which prints examples of it).

In [ ]:
examples = pd.concat([val.s2[val.s2[C.NAME].str.contains(r"[\x{0900}-\x{0DFF}]", regex=True)].head(3),
                      val.s3.sample(5, random_state=1)], ignore_index=True)
normalise_records(examples)[["entity_id", "name_norm", "name_core", "legal_form", "name_squash",
                             "addr_norm", "region"]].assign(raw_name=examples[C.NAME].to_numpy())

### 4.2 Blocking (`blocking.py`)

Candidate pairs are generated inside each country partition (whatever country values exist,
so France needs nothing special) as the union of:

| Bit | Pass | Why |
|---|---|---|
| 1 / 2 / 4 | exact `name_core` / `name_sorted` / `name_squash`, pool key groups ≤ 50 | cheap, exact; word swaps; domain/leet forms |
| 8 | name char 3-gram TF-IDF top-10 (cos ≥ 0.5) against pool records with ≤ 3 address tokens | typos where the address cannot help (empty / city-only addresses) |
| 16 | name + address word uni+bigram TF-IDF top-25 (cos ≥ 0.2, `max_df` 0.01) | the workhorse: renames, script names, same-name decoys ranked by their address |

Pairs are capped at 60 per S1 (exact pairs first, then by similarity). Retrieval uses
`sparse_dot_topn` (Apache-2.0) multi-threaded sparse top-k. Measured on 10k-S1 val samples
while designing it: running char 3-grams on every record costs ~3 ms per S1 (hours on test),
so that pass is restricted; word bigrams keep the address signal that `max_df` removes from
frequent unigrams (`rajendra nagar`, `5 52`). Recall is reported on the full val fold in §5.

In [ ]:
pd.Series({k: v for k, v in asdict(cfg.blocking).items()})

### 4.3 Pair features (`features.py`)

Similarity features per candidate pair, grouped as in 07 §2. Names rank, addresses decide:
many address-agreement features (token set / Jaccard / containment, house-number and number
set agreement, region, last tokens) sit next to fuzzy name scores (rapidfuzz, MIT) on the
normalised and core names, legal-form agreement, blocking similarities and the pair's
context inside its S1 group (rank and gap to the best candidate). No country feature (open
set). Every similarity is in [0, 1]; NaN where a field is empty (LightGBM handles it).

In [ ]:
pd.DataFrame([(g, ", ".join(FEATURE_COLUMNS[g])) for g in cfg.feature_groups],
             columns=["group", "features"]).assign(n=lambda d: d.features.str.count(",") + 1)

### 4.4 Matching model (as v101) and 4.5 decision rule (re-tuned)

`pipeline.retune` loads nothing new into the matcher: it scores the tune-side entities
against the whole train-fold pool (`tune_pool="train"`; the fit-fold records are decoys that
belong to other businesses, exactly like the unmatched part of the test pool) and re-runs the
3,906-rule grid plus refinement. The val fold is not touched.

In [ ]:
from entity_resolution.pipeline import Fitted, retune
fit_timings: dict[str, float] = {}
t0 = time.time()
parent = Fitted.load(C.EXPERIMENTS / "v101_name_frequency" / "artifacts", cfg)
fitted = retune(cfg, parent, train, ARTIFACTS, fit_timings)
timings["tune_seconds"] = fit_timings.get("tune_seconds", 0.0)
print(f"retune {time.time() - t0:.0f} s; stages: {fit_timings}")
print("v101 rule:", parent.rule)
print("v102 rule:", fitted.rule)

Blocking quality of the three training-time sides (recall, candidates per S1), the 20 most
important features (gain, normalised) and the best rules of the tune grid. An address
feature missing from the top 15 would be a bug (08 §8).

In [ ]:
blk = pd.DataFrame({side: fitted.info[f"{side}_blocking"] for side in ("fit", "stop", "tune", "tunedense") if f"{side}_blocking" in fitted.info}).T
display(blk[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]])
display(fitted.matcher.importance().head(20).rename("gain").to_frame())
fitted.tune_table.sort_values("f_beta", ascending=False).head(10)

## 5. Evaluation on the validation fold

`run_fold` blocks, scores and decides the val fold once with the frozen rule. Primary metric:
**macro F0.5 over all val S1 entities, singletons included** (`evaluate.score_pairs`, equal to
`metrics.breakdown`). Blocking quality: candidate pair recall, entity recall and the ceiling
F0.5 a perfect matcher would reach on these candidates.

In [ ]:
t0 = time.time()
metrics, val_pairs, val_scored, val_matches = run_fold(cfg, fitted, val)
for k in ("blocking_seconds", "score_seconds", "decide_seconds", "normalise_seconds"):
    timings[k] = metrics.get(k, 0.0)
print(f"run_fold {time.time() - t0:.0f} s")
pd.Series(metrics)

Recall of each blocking pass on the val fold (a pair can come from several passes) and the
standard slice report (11 §7): country, source, Indic-script pool names, ambiguous core
names, singletons, number of true matches, empty addresses.

In [ ]:
is_true = pair_in(val_pairs, val.pairs)          # candidate pair is a true pair
rows = []
for name, bit in PASS_BITS.items():
    in_pass = (val_pairs["pass"].to_numpy() & bit) != 0
    if in_pass.any():
        rows.append((name, in_pass.sum() / len(val.s1), (in_pass & is_true).sum() / len(val.pairs)))
display(pd.DataFrame(rows, columns=["pass", "pairs_per_s1", "recall"]))
s1n_val = load_normalised("train", (1,), cfg, val.s1[C.ENTITY_ID], fitted.token_map)
pooln_val = load_normalised("train", (2, 3), cfg, pd.concat([val.s2, val.s3])[C.ENTITY_ID],
                            fitted.token_map)
slices = slice_report(val_matches, val, s1n_val, pooln_val)
slices.to_csv(ARTIFACTS / "slices.csv", index=False)
slices

**Harder validation** (11 §6): test has 5.8 pool records per S1 against 4.7 in train, so the
same frozen rule is also scored on a val variant that drops 20 % of the S1 entities but keeps
their pool records (they become unowned decoys). A version whose harder score falls while
val rises is buying recall with false merges.

In [ ]:
harder_metrics, *_ = run_fold(cfg, fitted, harder_fold(val), tag="harder")
print({k: round(v, 4) for k, v in harder_metrics.items() if k.startswith("f_beta") or k.startswith("pair")})

## 6. Error analysis

Samples of the four error kinds, both records side by side with the model probability:
false merges on matched entities, missed true pairs, matched entities predicted empty
(false singletons) and predictions on true singletons. The counts say where the lost F0.5
sits; the samples name the pattern for the next version.

In [ ]:
counts = {}
for kind in ("false_merge", "missed", "false_singleton", "singleton_merge"):
    sample = error_samples(val_matches, val, kind, n=10, scored=val_scored)
    counts[kind] = len(error_samples(val_matches, val, kind, n=10**9))
    print(f"--- {kind}: {counts[kind]} pairs")
    display(sample)

## 7. Log the result

Records `metrics.json` and this version's row in `experiments/experiments.csv`, stamped
with the git commit of `src/` (it must not end in `-dirty`). Keys follow 13 §2.2.

In [ ]:
record = {
    "hypothesis": "a rule tuned at test-like decoy density raises precision where the leaderboard scores",
    "blocking_config": asdict(cfg.blocking), "feature_groups": list(cfg.feature_groups),
    "model_params": asdict(cfg.model), "rule": asdict(fitted.rule),
    **{k: metrics[k] for k in ("f_beta", "f_beta_singletons", "f_beta_matched",
                               "pair_precision", "pair_recall")},
    **{k: metrics[k] for k in ("cand_recall", "entity_recall", "ceiling_f_beta",
                               "cands_mean", "cands_p95")},
    "harder_f_beta": harder_metrics["f_beta"],
    "tune_f_beta": float(fitted.tune_table["f_beta"].max()),
    "n_fp": counts["false_merge"] + counts["singleton_merge"], "n_fn": counts["missed"],
    "n_false_singleton": counts["false_singleton"], "errors": counts,
    "token_map_size": fitted.info["token_map_size"],
    "best_iteration": fitted.matcher.best_iteration_, "parent_rule": asdict(parent.rule),
    **timings, "peak_rss_gb": peak_rss_gb(),
}
parent_m = json.loads((C.EXPERIMENTS / "v101_name_frequency" / "metrics.json").read_text())["metrics"]
DECISION = "INVESTIGATE"  # judged with the dense-val diagnostic (see §1), not plain val alone
record["decision"] = DECISION
print("parent v101:", parent_m["f_beta"], "->", record["f_beta"], "(plain val)")
row = log_result(
    EXP_DIR, change="v101 matcher; rule re-tuned against the whole train pool (tune_pool=train)",
    group="E2", local_f05=metrics["f_beta"], cand_recall=metrics["cand_recall"],
    notes=(f"cands {metrics['cands_mean']:.1f}/S1; singleton F0.5 "
           f"{metrics['f_beta_singletons']:.4f}; harder {harder_metrics['f_beta']:.4f}"),
    metrics=record, owner="M1", parent="v101", decision=DECISION)
row

## 8. Conclusion

Written after the run from the numbers above (see the markdown cell at the end of §9).

## 9. Test inference (shortlisted: upload #1)

Same pipeline on the test split: normalise (cached), block per country (France included),
score, decide with the frozen rule, write both files with `submission.write_pairs` from the
exact pairs frame that was scored. Then the sanity checks of 11 §10: one row per test S1 in
both files, every country present with candidates and matches, match rates and candidates per
S1 close to val.

In [ ]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test(cfg, fitted)
print(f"run_test {time.time() - t0:.0f} s -> {match_path}, {cand_path}")
import shutil
dest = C.ROOT / "submissions" / "v102"          # the uploaded files, kept per version
dest.mkdir(parents=True, exist_ok=True)
for path in (match_path, cand_path):
    shutil.copy2(path, dest / path.name)
print("copied to", dest)


def per_country(s1n, matches, n_cands_by_s1):
    """Match rate, matches and candidates per S1, by country (11 §10 sanity table)."""
    country = s1n.set_index(C.ENTITY_ID)[C.COUNTRY]
    n_s1 = s1n.groupby(C.COUNTRY).size()
    by = matches[C.S1_ID].map(country)
    return pd.DataFrame({
        "s1": n_s1,
        "cands_per_s1": n_cands_by_s1.groupby(n_cands_by_s1.index.map(country)).sum() / n_s1,
        "matched_share": matches.groupby(by)[C.S1_ID].nunique() / n_s1,
        "matches_per_s1": matches.groupby(by).size() / n_s1,
    })


test_table = per_country(s1n_test, test_matches, test_summary["n_cands"])
val_table = per_country(s1n_val, val_matches, val_pairs.groupby(C.S1_ID).size())
display(pd.concat({"test": test_table, "val": val_table}))
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
pd.crosstab(test_summary.index.map(country_of),
            pd.cut(test_summary["p_max"], [0, .1, .3, .5, .7, .9, 1.0]), normalize="index").round(3)

Both validators on the exact files that will be uploaded: ours (`submission.validate` with
id existence checks) and the organisers' stdlib-only `validate_submission.py`.

In [ ]:
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")